# <mark style="background-color: lightblue">Notebook made for the LSABD 2026-2027 course for dataset 2.</mark>
**Course:** Large Scale Analysis of Biomedical Data (LSABD), Ghent University  
**Dataset:** GSE80655: post-mortem brain RNA-seq from four psychiatric diagnostic groups  
**Scope:** This notebook includes all 4 steps of the datamining group project using human-readable code, additional information per step, and code information.


## Study design and dependence structure
The dataset contains 96 donors and 281 libraries. Most donors contributed tissue from three brain regions. A donor's libraries are repeated measurements from the same person; they must not be treated as 281 independent individuals in inferential analyses. Diagnosis, sex, age, ethnicity, PMI and brain pH are donor-level variables; brain region is a within-donor variable.


**Key sources:** LSABD Group Project Introduction (Dataset 2); LSABD Assessment Rubric 2026–2027 (Data Preprocessing and Exploration); Ramaker et al. (2017), *Genome Medicine*, 9:72, https://doi.org/10.1186/s13073-017-0458-5.



# <mark style="background-color: lightblue">Part 1 - Data exploration, RNA-seq quality control and preprocessing</mark>

### <span style="color: red;">Important source limitations + fix</span>
The given data by our professors contained the gene-count matrix and sample metadata, but they did not contain a gene-annotation table, percentage of reads uniquely aligned (PRUA), alignment-level mitochondrial-read percentage, or sequencing-batch variable. To make gene IDs interpretable, this notebook added the Ensembl GRCh37 release 75 gene annotation, which was specified by the original publication. *This annotation is an external reference; it does not change the supplied counts.*

The notebook also calculates a mitochondrial gene-count fraction from the annotated mitochondrial genes. This is only an exploratory count-matrix proxy. It is not the percentage of reads aligned to mitochondrial DNA and is also not a substitute for PRUA. We therefore do not claim to reproduce the paper's alignment-quality correction.

The supplied count matrix contains X- and Y-chromosome features according to the annotation. The main Part 1 analyses retain the supplied feature set. A separate X/Y-excluded copy is created for later genome-wide analyses that deliberately follow the paper's exclusion of sex-chromosome genes.

## 0. Environment, imports and project paths

This notebook is designed for **Python 3.11**. 

Run the notebook from the project directory. The supplied count and metadata CSV files must be in the subfolder `data`. The Ensembl GRCh37 release 75 GTF must be in the subfolder's subfolder `data/annotation` as `Homo_sapiens.GRCh37.75.gtf.gz` (which is the same when downloaded online, so the raw file). The results folders are created automatically. The code includes checks that stop execution when required inputs or validation criteria are missing.

In [ ]:
from __future__ import annotations

import sys
import re
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.decomposition import PCA

# PyDESeq2 is used for DESeq2-style size-factor normalization and VST.
from pydeseq2.dds import DeseqDataSet

PYDESEQ2_VERSION = version("pydeseq2")
if PYDESEQ2_VERSION != "0.5.4":
    raise RuntimeError(
        f"Expected pydeseq2==0.5.4, found {PYDESEQ2_VERSION}. "
        "Install the pinned requirements.txt in the Python 3.11 environment."
    )

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ANNOTATION_DIR = DATA_DIR / "annotation"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
PROCESSED_DIR = RESULTS_DIR / "processed"

for directory in [ANNOTATION_DIR, FIGURE_DIR, TABLE_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

COUNTS_PATH = DATA_DIR / "GSE80655_counts.csv"
METADATA_PATH = DATA_DIR / "GSE80655_metadata.csv"
GTF_PATH = ANNOTATION_DIR / "Homo_sapiens.GRCh37.75.gtf.gz"

missing_files = [str(path) for path in [COUNTS_PATH, METADATA_PATH] if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "The input CSV file(s) were not found:\n  - " + "\n  - ".join(missing_files) +
        "\nCopy both supplied CSV files into the project's data/ folder and rerun this cell."
    )

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.frameon": False,
})


def save_figure(figure: plt.Figure, file_stem: str) -> None:
    """Saves a figure as a PNG and PDF."""
    figure.savefig(FIGURE_DIR / f"{file_stem}.png", dpi=300, bbox_inches="tight")
    figure.savefig(FIGURE_DIR / f"{file_stem}.pdf", bbox_inches="tight")

# Output checklist.
print(f"Python interpreter : {sys.executable}")
print(f"Python version     : {sys.version.split()[0]}")
print(f"PyDESeq2 version   : {PYDESEQ2_VERSION}")
print(f"Project directory : {PROJECT_ROOT}")
print(f"Input directory   : {DATA_DIR}")
print(f"Results directory : {RESULTS_DIR}")

## 1. Load the data and validate the sample-to-metadata join
The count matrix is expected to have **genes in rows and libraries in columns**. The metadata contains a `description` column with the library IDs used as count-matrix column names. `GSM` is a GEO accession and is not the matching key for the count matrix.

We verify uniqueness, count validity, and exact sample matching before doing any analysis. We then reorder the metadata to match the count-matrix columns. This prevents a silent sample-ordering error from assigning the wrong diagnosis or brain region to an expression profile.

In [ ]:
counts_raw = pd.read_csv(COUNTS_PATH, index_col=0, low_memory=False)
metadata_raw = pd.read_csv(METADATA_PATH, low_memory=False)

# Gives the row index an explicit name and keep identifiers as strings.
counts_raw.index = counts_raw.index.astype(str)
counts_raw.columns = counts_raw.columns.astype(str)
counts_raw.index.name = "gene_id"

required_metadata_columns = {
    "GSM", "title", "description", "brain_region", "diagnosis", "age",
    "sex", "ethnicity", "PMI", "brain_pH", "donor_id"
}
missing_metadata_columns = required_metadata_columns.difference(metadata_raw.columns)
if missing_metadata_columns:
    raise ValueError(f"Metadata is missing required columns: {sorted(missing_metadata_columns)}")

metadata_raw["description"] = metadata_raw["description"].astype(str)

# Identifiers must be unique before a one-to-one alignment is possible.
assert counts_raw.index.is_unique, "Duplicate gene identifiers detected in the count matrix."
assert counts_raw.columns.is_unique, "Duplicate library identifiers detected in the count matrix."
assert metadata_raw["description"].is_unique, "Duplicate library IDs in metadata['description']."

# Checks that the matrix contains numeric, non-negative integer counts.
counts_numeric = counts_raw.apply(pd.to_numeric, errors="raise")
if counts_numeric.isna().any().any():
    raise ValueError("The count matrix contains missing values.")
if (counts_numeric.to_numpy() < 0).any():
    raise ValueError("Negative values were found; raw read counts cannot be negative.")
if not np.equal(counts_numeric.to_numpy(), np.floor(counts_numeric.to_numpy())).all():
    raise ValueError("Non-integer values were found; the input is not a raw integer count matrix.")

count_library_ids = set(counts_numeric.columns)
metadata_library_ids = set(metadata_raw["description"])
if count_library_ids != metadata_library_ids:
    only_in_counts = sorted(count_library_ids - metadata_library_ids)
    only_in_metadata = sorted(metadata_library_ids - count_library_ids)
    raise ValueError(
        "Count and metadata library IDs do not match. "
        f"Only in counts: {only_in_counts[:10]}; only in metadata: {only_in_metadata[:10]}"
    )

# Aligns metadata to the exact order of count-matrix columns.
metadata_indexed = metadata_raw.set_index("description", verify_integrity=True)
meta = metadata_indexed.loc[counts_numeric.columns].copy()
meta.index.name = "library_id"

# Uses integer storage only after checking that all values are valid integers.
counts = counts_numeric.astype(np.int64)

# Assigns categorical data types with an explicit, stable plotting order.
DIAGNOSIS_ORDER = ["Control", "Schizophrenia", "Bipolar Disorder", "Major Depression"]
REGION_ORDER = ["AnCg", "DLPFC", "nAcc"]

meta["diagnosis"] = pd.Categorical(meta["diagnosis"], categories=DIAGNOSIS_ORDER, ordered=True)
meta["brain_region"] = pd.Categorical(meta["brain_region"], categories=REGION_ORDER, ordered=True)
meta["sex"] = meta["sex"].astype("category")
meta["ethnicity"] = meta["ethnicity"].astype("category")
meta["donor_id"] = meta["donor_id"].astype(str)  # Identifier, not a continuous measurement.

if meta["diagnosis"].isna().any() or meta["brain_region"].isna().any():
    raise ValueError("Unexpected diagnosis or brain-region labels were found.")

assert list(counts.columns) == list(meta.index), "Count and metadata order is not identical."

# Output
print(f"Count matrix : {counts.shape[0]:,} genes × {counts.shape[1]} libraries")
print(f"Metadata     : {meta.shape[0]} libraries × {meta.shape[1]} variables")
print(f"Unique donors: {meta['donor_id'].nunique()}")
print(f"Total gene-level counts across all libraries: {counts.to_numpy(dtype=np.int64).sum():,}")
print(f"Fraction of all gene-by-library entries equal to zero: {(counts.to_numpy() == 0).mean():.1%}")

## 1.1 Add the original Ensembl GRCh37 release 75 gene annotation

The original paper states that reads were aligned and counted with STAR against genes annotated in GRCh37_E75. We therefore use the matching Ensembl release 75 GTF rather than a newer GRCh38 annotation or a newer annotation remapped to GRCh37. This preserves compatibility with the reference gene set used by the study.

The GTF is read directly from its compressed file. We retain one gene-level record per Ensembl gene ID and extract the gene symbol, chromosome/scaffold and gene biotype. The resulting annotation is then matched to the exact gene IDs and order in the supplied count matrix.

In [ ]:
# The original study used Ensembl GRCh37 release 75 (GRCh37_E75).
# Error raised when not found in the correct folder.
if not GTF_PATH.exists():
    raise FileNotFoundError(
        f"The Ensembl GRCh37 release 75 GTF was not found: {GTF_PATH}\n"
        "Download Homo_sapiens.GRCh37.75.gtf.gz from:\n"
        "https://ftp.ensembl.org/pub/release-75/gtf/homo_sapiens/\n"
        "and place it in the project's data/annotation/ folder."
    )

# Reads only the GTF columns required for gene-level annotation.
gtf_columns = [
    "seqname", "source", "feature", "start", "end",
    "score", "strand", "frame", "attributes"
]
gtf = pd.read_csv(
    GTF_PATH,
    sep="\t",
    comment="#",
    header=None,
    names=gtf_columns,
    usecols=[0, 2, 8],
    dtype={0: "string", 2: "string", 8: "string"},
    low_memory=False,
)

gene_rows = gtf.loc[
    gtf["feature"].eq("gene"), ["seqname", "attributes"]
].copy()
if gene_rows.empty:
    raise ValueError("No 'gene' feature rows were found in the Ensembl GTF.")


def extract_gtf_attribute(attributes: pd.Series, key: str) -> pd.Series:
    """Extracts one quoted key from a GTF attributes field."""
    pattern = rf'(?:^|;)\s*{re.escape(key)}\s+"([^"]+)"'
    return attributes.str.extract(pattern, expand=False)


annotation_raw = pd.DataFrame({
    "gene_id": extract_gtf_attribute(gene_rows["attributes"], "gene_id"),
    "symbol": extract_gtf_attribute(gene_rows["attributes"], "gene_name"),
    "chr": gene_rows["seqname"].astype("string"),
    "biotype": extract_gtf_attribute(gene_rows["attributes"], "gene_biotype"),
}).dropna(subset=["gene_id"])

if annotation_raw.empty:
    raise ValueError("Could not extract Ensembl gene IDs from the GTF attributes.")

# A stable Ensembl gene ID can occur on an alternative haplotype as well as its primary chromosome.
for column in ["symbol", "biotype"]:
    inconsistent = (
        annotation_raw.groupby("gene_id")[column]
        .nunique(dropna=False)
        .loc[lambda values: values > 1]
    )
    if not inconsistent.empty:
        raise ValueError(
            f"Conflicting {column} annotations were found for gene IDs: "
            f"{inconsistent.index[:10].tolist()}"
        )

standard_chromosomes = {str(number) for number in range(1, 23)} | {"X", "Y", "MT"}
primary_rows = annotation_raw[annotation_raw["chr"].isin(standard_chromosomes)]
multiple_primary_locations = (
    primary_rows.groupby("gene_id")["chr"].nunique().loc[lambda values: values > 1]
)
if not multiple_primary_locations.empty:
    raise ValueError(
        "Some Ensembl gene IDs map to more than one primary chromosome: "
        f"{multiple_primary_locations.index[:10].tolist()}"
    )

chromosome_counts_per_gene = annotation_raw.groupby("gene_id")["chr"].nunique()
primary_gene_ids = set(primary_rows["gene_id"].unique())
ambiguous_alt_only_ids = set(
    chromosome_counts_per_gene.loc[chromosome_counts_per_gene > 1].index
) - primary_gene_ids
annotation_raw.loc[
    annotation_raw["gene_id"].isin(ambiguous_alt_only_ids), "chr"
] = "ALT_MULTIPLE"

annotation_raw["is_primary_chromosome"] = annotation_raw["chr"].isin(standard_chromosomes)
annotation_raw = annotation_raw.sort_values(
    ["gene_id", "is_primary_chromosome", "chr"],
    ascending=[True, False, True],
)

gene_annotation = (
    annotation_raw.drop_duplicates(subset="gene_id")
    .drop(columns="is_primary_chromosome")
    .set_index("gene_id")
)
gene_annotation.index = gene_annotation.index.astype(str)
gene_annotation.index.name = "gene_id"

# Requires complete annotation coverage for every gene in the supplied matrix.
missing_annotation_ids = counts.index.difference(gene_annotation.index)
if len(missing_annotation_ids) > 0:
    raise ValueError(
        f"{len(missing_annotation_ids):,} count-matrix genes are absent from "
        "the Ensembl GRCh37 release 75 GTF. Examples: "
        f"{missing_annotation_ids[:10].tolist()}"
    )

gene_annotation = gene_annotation.reindex(counts.index)
assert gene_annotation.index.is_unique
assert list(gene_annotation.index) == list(counts.index)

# Identifies EGR1 from its gene symbol instead of hardcoding the ID.
egr1_ids = gene_annotation.index[gene_annotation["symbol"].eq("EGR1")].tolist()
if len(egr1_ids) != 1:
    raise ValueError(f"Expected exactly one EGR1 gene ID, found: {egr1_ids}")
EGR1_ENSEMBL_ID = egr1_ids[0]
if EGR1_ENSEMBL_ID != "ENSG00000120738":
    raise ValueError(
        "EGR1 did not map to the expected Ensembl stable ID "
        f"ENSG00000120738; found {EGR1_ENSEMBL_ID}. Check the annotation source."
    )

# Calculates a mitochondrial gene-count fraction for each library.
mt_gene_ids = gene_annotation.index[gene_annotation["chr"].eq("MT")]
if len(mt_gene_ids) == 0:
    raise ValueError("No mitochondrial genes (chr == 'MT') were found in the annotation.")

total_gene_counts_per_library = counts.sum(axis=0)
if (total_gene_counts_per_library <= 0).any():
    raise ValueError("At least one library has zero total gene-level counts.")

mt_gene_count_fraction = (
    counts.loc[mt_gene_ids].sum(axis=0) / total_gene_counts_per_library
).rename("mt_gene_count_fraction")

# Descriptive summary and explicit check of sex-chromosome features.
chromosome_gene_counts = gene_annotation["chr"].value_counts()
print(f"Annotation source: Ensembl GRCh37 release 75")
print(f"Count-matrix genes annotated: {gene_annotation.shape[0]:,} / {counts.shape[0]:,}")
print(f"EGR1: {EGR1_ENSEMBL_ID}")
print(f"Mitochondrial genes in count matrix: {len(mt_gene_ids)}")
print(
    "X- and Y-chromosome genes in count matrix: "
    f"{int(gene_annotation['chr'].eq('X').sum()):,} X; "
    f"{int(gene_annotation['chr'].eq('Y').sum()):,} Y"
)
print("Mitochondrial gene-count fraction by library:")
display(mt_gene_count_fraction.describe().to_frame().T)

# Retains the annotation in count-matrix row order for all later project parts.
display(gene_annotation.head())

## 2. Describe the variables and study design
The project introduction asks in part 1 to identify the number of samples and features and classify variables by type. We distinguish identifiers from actual measurements.

In [ ]:
variable_roles = {
    "GSM": "Identifier (GEO accession)",
    "title": "Composite sample identifier / descriptive text",
    "description": "Identifier (library ID; count-matrix join key)",
    "brain_region": "Nominal categorical (within-donor tissue region)",
    "diagnosis": "Nominal categorical (donor-level outcome grouping)",
    "age": "Continuous (years; donor-level)",
    "sex": "Nominal categorical (donor-level)",
    "ethnicity": "Nominal categorical (donor-level)",
    "PMI": "Continuous (hours; donor-level)",
    "brain_pH": "Continuous (donor-level; tissue-quality proxy)",
    "donor_id": "Identifier / grouping variable (nominal)",
}

metadata_variable_summary = pd.DataFrame({
    "variable": metadata_raw.columns,
    "role_and_type": [variable_roles.get(column, "Review variable definition") for column in metadata_raw.columns],
    "stored_dtype": [meta[column].dtype.name if column in meta.columns else metadata_raw[column].dtype.name
                      for column in metadata_raw.columns],
    "number_of_unique_values": [metadata_raw[column].nunique(dropna=False) for column in metadata_raw.columns],
    "missing_values": [metadata_raw[column].isna().sum() for column in metadata_raw.columns],
})

display(metadata_variable_summary)
metadata_variable_summary.to_csv(TABLE_DIR / "01_metadata_variable_summary.csv", index=False)

# Samples distribution across diagnostic groups and brain regions.
sample_distribution = pd.crosstab(meta["diagnosis"], meta["brain_region"], margins=True, dropna=False)
print("Libraries per diagnosis * brain region:")
display(sample_distribution)

# One row per donor: counts the number of libraries and regions contributed.
donor_summary = meta.groupby("donor_id", observed=True).agg(
    n_libraries=("diagnosis", "size"),
    n_regions=("brain_region", "nunique"),
    diagnosis=("diagnosis", "first"),
    age=("age", "first"),
    sex=("sex", "first"),
    ethnicity=("ethnicity", "first"),
    PMI=("PMI", "first"),
    brain_pH=("brain_pH", "first"),
)

print("Number of libraries contributed per donor:")
display(donor_summary["n_libraries"].value_counts().sort_index().rename_axis("libraries_per_donor").to_frame("donor_count"))
print(f"Donors with all three regions: {(donor_summary['n_regions'] == 3).sum()}")
print(f"Donors with two regions:       {(donor_summary['n_regions'] == 2).sum()}")

# Checks whether donor-level variables are constant across repeated libraries.
donor_level_variables = ["diagnosis", "age", "sex", "ethnicity", "PMI", "brain_pH"]
within_donor_unique = meta.groupby("donor_id", observed=True)[donor_level_variables].nunique(dropna=False)
donor_consistency = (within_donor_unique <= 1).all(axis=0).rename("constant_within_every_donor")
print("Are donor-level variables constant within each donor?")
display(donor_consistency.to_frame())

# Number of donors per diagnosis (each donor should belong to one diagnosis).
donors_by_diagnosis = donor_summary.groupby("diagnosis", observed=True).size().rename("number_of_donors")
print("Donors per diagnosis:")
display(donors_by_diagnosis.to_frame())

## 3. Missing values and how they are handled
Missingness must be described before deciding whether to remove anything. The only missing variable in the supplied metadata is `brain_pH`.

The three missing pH values belong to the same donor. For Part 1, we do not impute pH and we do not discard the count libraries. Normalization and blind PCA do not require pH. A later model that explicitly includes brain pH should use a documented complete-case strategy and exclude all three libraries from this one donor, rather than retaining some regions from that donor or inventing a pH value.

In [ ]:
missingness_table = pd.DataFrame({
    "missing_count": metadata_raw.isna().sum(),
    "missing_percent": (metadata_raw.isna().mean() * 100).round(2),
})
print("Missingness in the supplied metadata:")
display(missingness_table[missingness_table["missing_count"] > 0])
missingness_table.to_csv(TABLE_DIR / "01_missingness.csv")

missing_ph_rows = meta.loc[meta["brain_pH"].isna(), [
    "GSM", "brain_region", "diagnosis", "donor_id", "age", "PMI"
]]
print("Libraries with missing brain pH:")
display(missing_ph_rows)
print(f"Missing-pH libraries: {len(missing_ph_rows)}")
print(f"Unique donors affected: {missing_ph_rows['donor_id'].nunique()}")

complete_case_pH = meta.loc[meta["brain_pH"].notna()].copy()
print(
    f"Complete-case subset for a future pH-adjusted model: "
    f"{len(complete_case_pH)} libraries from {complete_case_pH['donor_id'].nunique()} donors."
)

# Descriptive donor-level clinical covariates (one row per donor).
donor_numeric_summary = donor_summary.groupby("diagnosis", observed=True)[
    ["age", "PMI", "brain_pH"]
].agg(["count", "median", "min", "max"])
print("Donor-level clinical covariates by diagnosis (descriptive only):")
display(donor_numeric_summary.round(2))

print("Sex distribution by diagnosis (donor-level):")
display(pd.crosstab(donor_summary["diagnosis"], donor_summary["sex"], margins=True))
print("Ethnicity distribution by diagnosis (donor-level):")
display(pd.crosstab(donor_summary["diagnosis"], donor_summary["ethnicity"], margins=True))

## 4. Raw-count quality control at the library level
The supplied count matrix allows us to calculate these sample-level metrics:
- Total gene-level counts: sum of counts assigned to all genes in a library. This is a useful depth/composition indicator, but it is not exactly the total number of sequenced reads.
- Detected genes: number of genes with at least one count.
- Genes with at least 10 counts: a  measure of how many features have more than minimal support.
- Zero fraction: proportion of gene-by-library entries that are zero.

We use robust median/MAD scores to flag unusually extreme values within each brain region, so expected between-region differences do not dominate the outlier screen.
- Mitochondrial gene-count fraction: the fraction of assigned gene counts attributed to annotated mitochondrial genes.

In [ ]:
sample_qc = pd.DataFrame(index=counts.columns)
sample_qc.index.name = "library_id"
sample_qc["total_gene_counts"] = counts.sum(axis=0)
sample_qc["detected_genes"] = (counts > 0).sum(axis=0)
sample_qc["genes_at_least_10_counts"] = (counts >= 10).sum(axis=0)
sample_qc["zero_fraction"] = (counts == 0).mean(axis=0)
sample_qc["log10_total_gene_counts"] = np.log10(sample_qc["total_gene_counts"])
sample_qc = sample_qc.join(meta[["diagnosis", "brain_region", "donor_id", "brain_pH", "age", "PMI"]])
sample_qc["mt_gene_count_fraction"] = mt_gene_count_fraction.reindex(sample_qc.index)


def robust_modified_z_score(values: pd.Series) -> pd.Series:
    """Returns median/MAD-based modified z-scores; safe when MAD is zero."""
    median_value = values.median()
    median_absolute_deviation = (values - median_value).abs().median()
    if not np.isfinite(median_absolute_deviation) or median_absolute_deviation == 0:
        return pd.Series(0.0, index=values.index)
    return 0.6745 * (values - median_value) / median_absolute_deviation


sample_qc["robust_z_total_counts"] = sample_qc.groupby("brain_region", observed=True)[
    "log10_total_gene_counts"
].transform(robust_modified_z_score)
sample_qc["robust_z_detected_genes"] = sample_qc.groupby("brain_region", observed=True)[
    "detected_genes"
].transform(robust_modified_z_score)
sample_qc["robust_z_zero_fraction"] = sample_qc.groupby("brain_region", observed=True)[
    "zero_fraction"
].transform(robust_modified_z_score)

OUTLIER_SCREEN_THRESHOLD = 3.5
sample_qc["flag_total_counts"] = sample_qc["robust_z_total_counts"].abs() > OUTLIER_SCREEN_THRESHOLD
sample_qc["flag_detected_genes"] = sample_qc["robust_z_detected_genes"].abs() > OUTLIER_SCREEN_THRESHOLD
sample_qc["flag_zero_fraction"] = sample_qc["robust_z_zero_fraction"].abs() > OUTLIER_SCREEN_THRESHOLD
sample_qc["qc_flag_for_inspection"] = sample_qc[[
    "flag_total_counts", "flag_detected_genes", "flag_zero_fraction"
]].any(axis=1)

print("Summary of raw-count QC metrics:")
display(sample_qc[[
    "total_gene_counts", "detected_genes", "genes_at_least_10_counts",
    "zero_fraction", "mt_gene_count_fraction"
]].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(3))

qc_candidates = sample_qc.loc[sample_qc["qc_flag_for_inspection"]].sort_values("total_gene_counts")
print(f"Samples flagged for inspection (not automatic exclusions): {len(qc_candidates)}")
display(qc_candidates[[
    "total_gene_counts", "detected_genes", "zero_fraction", "mt_gene_count_fraction",
    "diagnosis", "brain_region", "donor_id", "flag_total_counts",
    "flag_detected_genes", "flag_zero_fraction"
]])

# Saves the full QC table so candidates can be checked later.
sample_qc.to_csv(TABLE_DIR / "01_sample_qc_metrics.csv")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.histplot(sample_qc["log10_total_gene_counts"], bins=25, kde=True, color="steelblue", ax=axes[0, 0])
axes[0, 0].set_title("Distribution of total gene-level counts")
axes[0, 0].set_xlabel("log10(total gene-level counts)")
axes[0, 0].set_ylabel("Number of libraries")

sns.scatterplot(
    data=sample_qc,
    x="total_gene_counts",
    y="detected_genes",
    hue="brain_region",
    style="diagnosis",
    s=55,
    alpha=0.85,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Detected genes versus total gene-level counts")
axes[0, 1].set_xlabel("Total gene-level counts")
axes[0, 1].set_ylabel("Number of detected genes")

sns.boxplot(
    data=sample_qc,
    x="brain_region",
    y="log10_total_gene_counts",
    hue="diagnosis",
    order=REGION_ORDER,
    ax=axes[1, 0],
)
axes[1, 0].set_title("Gene-level count depth by region and diagnosis")
axes[1, 0].set_xlabel("Brain region")
axes[1, 0].set_ylabel("log10(total gene-level counts)")

sns.boxplot(
    data=sample_qc,
    x="diagnosis",
    y="zero_fraction",
    hue="brain_region",
    order=DIAGNOSIS_ORDER,
    ax=axes[1, 1],
)
axes[1, 1].set_title("Zero fraction by diagnosis and region")
axes[1, 1].set_xlabel("Diagnosis")
axes[1, 1].set_ylabel("Fraction of zero counts")
axes[1, 1].tick_params(axis="x", rotation=18)

fig.suptitle("Raw RNA-seq library quality-control overview", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "01_sample_qc")
plt.show()

## 5. Filter genes with very low expression
Genes with almost no counts provide little information for expression comparisons and can make later analyses unnecessarily large. We use a global, diagnosis-independent filter:

> **Keep a gene if it has at least 10 counts in at least 10 libraries.**

This is a deliberately permissive pre-filter. It uses no diagnosis labels, region-specific expression differences or test results. We also display how the retained gene count changes under stricter prevalence thresholds, so the impact of the choice is transparent.

### Separate X/Y-excluded matrix for later genome-wide analyses
The original publication states that X- and Y-chromosome genes were excluded from its genome-wide analyses. The Ensembl annotation allows us to identify those genes explicitly in the supplied matrix. We therefore create a separate filtered raw-count matrix without X/Y genes.

**Important:** this does not overwrite `counts`, `counts_filtered`.

The EGR1 identifier is obtained from the annotation by gene symbol, rather than hard-coded. EGR1 is autosomal (chromosome 5), so it remains in the X/Y-excluded matrix.

In [ ]:
MIN_COUNT = 10
MIN_SAMPLES = 10

number_of_libraries_with_minimum_count = (counts >= MIN_COUNT).sum(axis=1)
keep_gene = number_of_libraries_with_minimum_count >= MIN_SAMPLES
counts_filtered = counts.loc[keep_gene].copy()

filter_sensitivity_rows = []
for required_libraries in [5, 10, 20, 29, 57]:
    passes = number_of_libraries_with_minimum_count >= required_libraries
    filter_sensitivity_rows.append({
        "minimum_count_per_library": MIN_COUNT,
        "minimum_number_of_libraries": required_libraries,
        "retained_genes": int(passes.sum()),
        "removed_genes": int((~passes).sum()),
    })

gene_filter_sensitivity = pd.DataFrame(filter_sensitivity_rows)
print(f"Genes before filtering: {counts.shape[0]:,}")
print(f"Genes after filtering : {counts_filtered.shape[0]:,}")
print(f"Genes removed         : {(~keep_gene).sum():,}")
print("Sensitivity to the minimum prevalence threshold:")
display(gene_filter_sensitivity)
gene_filter_sensitivity.to_csv(TABLE_DIR / "01_gene_filter_sensitivity.csv", index=False)

# EGR1_ENSEMBL_ID was identified by gene symbol in the Ensembl annotation section.
if EGR1_ENSEMBL_ID in counts.index:
    egr1_raw = counts.loc[EGR1_ENSEMBL_ID]
    print("EGR1 identifier is present in the supplied matrix.")
    print(
        "Raw EGR1 counts (min / median / max): "
        f"{egr1_raw.min():,} / {egr1_raw.median():,.1f} / {egr1_raw.max():,}"
    )
    print(f"EGR1 passes the global expression filter: {EGR1_ENSEMBL_ID in counts_filtered.index}")
else:
    print(
        "Oopsie, the expected EGR1 Ensembl identifier was not found. -_-"
        "Do not guess a replacement identifier; obtain/verify the appropriate gene annotation."
    )

assert counts_filtered.shape[0] > 0, "Oopsie, the expression filter unexpectedly removed every gene. -_-"
assert counts_filtered.index.is_unique and counts_filtered.columns.is_unique
assert (counts_filtered.to_numpy() >= 0).all()
assert np.equal(counts_filtered.to_numpy(), np.floor(counts_filtered.to_numpy())).all()

# Creates a separate X/Y-excluded matrix for later genome-wide analyses.
sex_chromosome_gene_ids = gene_annotation.index[
    gene_annotation["chr"].isin(["X", "Y"])
]
sex_chromosome_gene_ids_filtered = counts_filtered.index[
    counts_filtered.index.isin(sex_chromosome_gene_ids)
]

if len(sex_chromosome_gene_ids) == 0:
    raise ValueError(
        "No X/Y genes were identified. Check the chromosome labels in the GTF annotation."
    )

counts_filtered_no_sex_chr = counts_filtered.drop(
    index=sex_chromosome_gene_ids_filtered
).copy()
excluded_sex_chromosome_annotation = gene_annotation.loc[
    sex_chromosome_gene_ids_filtered,
    ["symbol", "chr", "biotype"],
].copy()

# Validation: only the separate downstream copy excludes X/Y.
assert counts_filtered_no_sex_chr.shape[1] == counts_filtered.shape[1]
assert list(counts_filtered_no_sex_chr.columns) == list(counts_filtered.columns)
assert counts_filtered_no_sex_chr.index.is_unique
assert not gene_annotation.loc[
    counts_filtered_no_sex_chr.index, "chr"
].isin(["X", "Y"]).any()
assert counts_filtered_no_sex_chr.shape[0] + len(sex_chromosome_gene_ids_filtered) == counts_filtered.shape[0]

# EGR1 is autosomal and should remain available for Part 2.
assert gene_annotation.loc[EGR1_ENSEMBL_ID, "chr"] not in ["X", "Y"]
assert EGR1_ENSEMBL_ID in counts_filtered_no_sex_chr.index

n_x_filtered = int(excluded_sex_chromosome_annotation["chr"].eq("X").sum())
n_y_filtered = int(excluded_sex_chromosome_annotation["chr"].eq("Y").sum())
print("Separate downstream matrix excluding sex chromosomes:")
print(f"  X-linked genes excluded after expression filtering: {n_x_filtered:,}")
print(f"  Y-linked genes excluded after expression filtering: {n_y_filtered:,}")
print(f"  Genes retained in main filtered matrix: {counts_filtered.shape[0]:,}")
print(f"  Genes retained in X/Y-excluded matrix: {counts_filtered_no_sex_chr.shape[0]:,}")
print("  Main Part 1 matrix remains unchanged: yes")


## 6. Normalize counts and calculate the variance-stabilizing transformation
### Why normalization is necessary
Raw read counts depend partly on library depth and RNA-seq composition. Comparing raw counts directly across libraries can therefore make differences in sequencing depth look like differences in gene expression.

We use the DESeq2 median-of-ratios size-factor method through PyDESeq2. Each library receives a size factor that estimates its relative sequencing depth/composition. Dividing counts by this factor produces normalized counts. The supplied matrix contains genes with non-zero counts in every library, so the standard ratio calculation has features from which to estimate geometric means (check their library info for more information).

### Why apply VST (variance-stabilizing transformation)?
Normalized counts remain heteroscedastic: genes with higher average expression tend to have larger count variance. The variance-stabilizing transformation uses the fitted mean–dispersion relationship to produce a log-like expression scale with a flatter mean–variance relationship. This is useful for PCA, clustering and continuous-expression analyses.

The VST is fitted with an intercept-only design (~1) and use_design=False. This is a blind, unsupervised transformation: diagnosis and region labels are not used to fit the transformation. All 281 libraries are retained here, including the one donor with missing pH, because neither size-factor estimation nor this transformation requires pH.

In [ ]:
# PyDESeq2 expects samples in rows and genes in columns.
counts_for_pydeseq2 = counts_filtered.T.copy()
normalization_metadata = pd.DataFrame(index=counts_for_pydeseq2.index)
normalization_metadata["diagnosis"] = meta.loc[counts_for_pydeseq2.index, "diagnosis"].astype(str)

assert list(counts_for_pydeseq2.index) == list(normalization_metadata.index)
assert counts_for_pydeseq2.index.is_unique

# Uses the DESeq2 median-of-ratios approach. The model is intercept-only because
# this stage estimates normalization/transformation, not a diagnosis effect.
# Genes positive in every library provide valid geometric means for the ratio method.
number_of_genes_positive_in_every_library = int((counts_filtered > 0).all(axis=1).sum())
print(
    "Genes with non-zero counts in every library (available for ratio factors): "
    f"{number_of_genes_positive_in_every_library:,}"
)

dds = DeseqDataSet(
    counts=counts_for_pydeseq2,
    metadata=normalization_metadata,
    design="~1",
    size_factors_fit_type="ratio",
    n_cpus=1,
    quiet=True,
)

dds.fit_size_factors()
size_factors = pd.Series(
    dds.obs["size_factors"].to_numpy(),
    index=dds.obs_names,
    name="size_factor",
)

# Calculates the blind VST. Transformed values are stored as samples * genes.
dds.vst(use_design=False)
vst_samples_by_genes = np.asarray(dds.layers["vst_counts"])
vst = pd.DataFrame(
    vst_samples_by_genes,
    index=dds.obs_names,
    columns=dds.var_names,
).T
vst.index.name = "gene_id"
vst.columns.name = "library_id"

# Reorders factors exactly to the original count-matrix column order.
size_factors = size_factors.reindex(counts_filtered.columns)

# Normalized counts are useful for QC and for checking the transformation.
normalized_counts = counts_filtered.div(size_factors, axis=1)

assert vst.shape == counts_filtered.shape, "VST dimensions do not match the filtered count matrix."
assert list(vst.index) == list(counts_filtered.index)
assert list(vst.columns) == list(counts_filtered.columns)
assert np.isfinite(vst.to_numpy()).all(), "Non-finite values found in VST matrix."
assert np.isfinite(size_factors.to_numpy()).all() and (size_factors > 0).all()

vst_no_sex_chr = vst.loc[counts_filtered_no_sex_chr.index].copy()
assert list(vst_no_sex_chr.index) == list(counts_filtered_no_sex_chr.index)
assert not gene_annotation.loc[vst_no_sex_chr.index, "chr"].isin(["X", "Y"]).any()

print("Size-factor summary:")
display(size_factors.describe().to_frame().round(3))
print(f"Geometric mean of size factors: {np.exp(np.log(size_factors).mean()):.4f}")
print(f"Filtered raw-count matrix: {counts_filtered.shape[0]:,} genes × {counts_filtered.shape[1]} libraries")
print(f"VST expression matrix     : {vst.shape[0]:,} genes × {vst.shape[1]} libraries")
print(f"VST numerical range       : {vst.to_numpy().min():.3f} to {vst.to_numpy().max():.3f}")

sample_qc["size_factor"] = size_factors
# Rewrites the QC table so it also contains the post-normalization size factors.
sample_qc.to_csv(TABLE_DIR / "01_sample_qc_metrics.csv")
size_factors.to_csv(TABLE_DIR / "01_size_factors.csv", header=True)

## 7. Check the effect of normalization and VST
A useful transformation should reduce the dependence of variability on average expression. The two panels compare the mean–standard-deviation relationship before and after VST. 
*This is a diagnostic rather than a formal proof that all gene-wise variances are identical.*

In [ ]:
normalized_mean = normalized_counts.mean(axis=1)
normalized_sd = normalized_counts.std(axis=1)
vst_mean = vst.mean(axis=1)
vst_sd = vst.std(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

axes[0].scatter(
    np.log10(normalized_mean + 0.1),
    np.log10(normalized_sd + 0.1),
    s=7,
    alpha=0.25,
)
axes[0].set_title("After size-factor normalization")
axes[0].set_xlabel("log10(mean normalized count + 0.1)")
axes[0].set_ylabel("log10(standard deviation + 0.1)")

axes[1].scatter(vst_mean, vst_sd, s=7, alpha=0.25)
axes[1].set_title("After variance-stabilizing transformation")
axes[1].set_xlabel("Mean VST expression")
axes[1].set_ylabel("Standard deviation of VST expression")

fig.suptitle("Mean–variance relationship before and after VST", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "02_normalization_mean_variance")
plt.show()

## 8. Principal component analysis (PCA)
PCA summarizes the largest patterns of variation across libraries. We first select the 2000 genes with the largest variance in the VST matrix, then perform PCA on the libraries. The data are centred by PCA; we do not scale each gene to unit variance, because doing so would give low-variance genes the same weight as highly variable genes.

We colour the same PCA projection in three ways:

1. **Brain region:** tests visually whether region is a major source of expression variation.
2. **Diagnosis:** checks whether diagnostic groups visibly separate or overlap.
3. **Brain pH:** checks whether a tissue-quality variable appears aligned with the leading expression axes.

*The points are libraries, not independent donors. PCA is therefore an exploratory visualization, not an inferential test. The original paper found strong region-associated structure and correlations between some PCs and sequencing-quality measures. We cannot reproduce its PRUA/mitochondrial-read correction because those variables are absent from the supplied files, as mentioned earlier.*

In [ ]:
N_TOP_VARIABLE_GENES = min(2000, vst.shape[0])
top_variable_gene_ids = vst.var(axis=1).nlargest(N_TOP_VARIABLE_GENES).index

pca_model = PCA(n_components=2, svd_solver="randomized", random_state=RANDOM_SEED)
pca_coordinates = pca_model.fit_transform(vst.loc[top_variable_gene_ids].T)

pca_scores = pd.DataFrame(
    pca_coordinates,
    index=vst.columns,
    columns=["PC1", "PC2"],
)
pca_scores.index.name = "library_id"
pca_scores = pca_scores.join(meta)
pca_scores = pca_scores.join(sample_qc[[
    "total_gene_counts", "detected_genes", "zero_fraction", "size_factor"
]])

explained_variance_percent = pca_model.explained_variance_ratio_ * 100
print(f"PC1 explains {explained_variance_percent[0]:.2f}% of variance.")
print(f"PC2 explains {explained_variance_percent[1]:.2f}% of variance.")
print(f"Combined variance explained by PC1 and PC2: {explained_variance_percent.sum():.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(17, 5.1))

# Panel 1: brain region.
for region in REGION_ORDER:
    subset = pca_scores.loc[pca_scores["brain_region"] == region]
    axes[0].scatter(
        subset["PC1"], subset["PC2"], s=30, alpha=0.8, label=region,
        edgecolors="white", linewidths=0.3,
    )
axes[0].set_title("PCA coloured by brain region")
axes[0].legend(title="Brain region")

# Panel 2: diagnosis.
for diagnosis in DIAGNOSIS_ORDER:
    subset = pca_scores.loc[pca_scores["diagnosis"] == diagnosis]
    axes[1].scatter(
        subset["PC1"], subset["PC2"], s=30, alpha=0.8, label=diagnosis,
        edgecolors="white", linewidths=0.3,
    )
axes[1].set_title("PCA coloured by diagnosis")
axes[1].legend(title="Diagnosis", fontsize=8)

# Panel 3: continuous brain pH. Missing pH values are simply not coloured.
valid_ph = pca_scores["brain_pH"].notna()
points = axes[2].scatter(
    pca_scores.loc[valid_ph, "PC1"],
    pca_scores.loc[valid_ph, "PC2"],
    c=pca_scores.loc[valid_ph, "brain_pH"],
    cmap="viridis",
    s=30,
    alpha=0.85,
    edgecolors="white",
    linewidths=0.25,
)
fig.colorbar(points, ax=axes[2], label="Brain pH")
axes[2].set_title("PCA coloured by brain pH")

for axis in axes:
    axis.set_xlabel(f"PC1 ({explained_variance_percent[0]:.1f}%)")
    axis.set_ylabel(f"PC2 ({explained_variance_percent[1]:.1f}%)")

fig.suptitle(
    f"PCA of the {N_TOP_VARIABLE_GENES:,} most variable VST genes",
    fontsize=14,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "03_pca_exploration")
plt.show()

pca_scores.to_csv(TABLE_DIR / "01_pca_scores.csv")

## 9. Explore donor-level covariates and potential confounding
Age, sex, ethnicity, PMI and brain pH are donor-level characteristics. To compare their distributions across diagnosis without counting a donor multiple times, this section uses the `donor_summary` table (96 rows), not the 281-library table.

*The plots and cross-tabulations are descriptive. We deliberately do not run ordinary ANOVA or Kruskal-Wallis tests on all 281 libraries, because that would ignore repeated measurements from the same donor.*

In [ ]:
donor_plot_data = donor_summary.reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for axis, variable, label in zip(
    axes,
    ["age", "PMI", "brain_pH"],
    ["Age at death (years)", "Post-mortem interval (hours)", "Brain pH"],
):
    sns.boxplot(
        data=donor_plot_data,
        x="diagnosis",
        y=variable,
        order=DIAGNOSIS_ORDER,
        color="lightsteelblue",
        showfliers=False,
        ax=axis,
    )
    sns.stripplot(
        data=donor_plot_data,
        x="diagnosis",
        y=variable,
        order=DIAGNOSIS_ORDER,
        color="black",
        alpha=0.55,
        size=3,
        jitter=0.18,
        ax=axis,
    )
    axis.set_title(label)
    axis.set_xlabel("Diagnosis")
    axis.set_ylabel(label)
    axis.tick_params(axis="x", rotation=25)

fig.suptitle("Donor-level covariate distributions by diagnosis", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "04_donor_level_covariates")
plt.show()

# Numerical summaries use one observation per donor.
donor_covariate_summary = donor_summary.groupby("diagnosis", observed=True)[
    ["age", "PMI", "brain_pH"]
].agg(["median", "mean", "std", "min", "max"])

donor_sex_table = pd.crosstab(donor_summary["diagnosis"], donor_summary["sex"])
donor_ethnicity_table = pd.crosstab(donor_summary["diagnosis"], donor_summary["ethnicity"])

print("Donor-level continuous covariates:")
display(donor_covariate_summary.round(2))
print("Donor-level sex distribution:")
display(donor_sex_table)
print("Donor-level ethnicity distribution:")
display(donor_ethnicity_table)

donor_summary.to_csv(TABLE_DIR / "01_donor_level_summary.csv")

## 10. What the exploratory analysis cannot conclude
### Current analysis limitations

1. The supplied files do not contain PRUA, alignment-level mitochondrial-read percentage, or sequencing-batch information. The mitochondrial gene-count fraction calculated above is only a count-matrix proxy and cannot reproduce the paper's PRUA or mitochondrial-alignment corrections.
2. Gene annotation is added from Ensembl GRCh37 release 75, the gene reference specified in the original paper. It provides gene symbols, chromosome/scaffold assignments and gene biotypes; it does not change the supplied raw counts.
3. The supplied count matrix includes genes assigned to chromosomes X and Y. The paper states that X- and Y-chromosome genes were excluded from its genome-wide analyses. We retained all supplied genes in the main Part 1 QC, normalization and PCA, and save a separate low-expression-filtered raw-count matrix without X/Y genes for downstream analyses that follow that feature scope. This is an explicit analysis-ready copy, not a claim that the original supplied matrix was already X/Y-filtered.
4. The pH-missing donor is retained for blind count normalization and PCA, but all three libraries should be omitted from a later model that requires pH unless a defensible value is obtained.
5. PCA and QC plots are exploratory; they do not account for within-donor correlation through a formal statistical model.
6. The VST transformation is intended for visualization and continuous-expression analyses. Negative-binomial models in later parts must start from raw integer counts.
7. For a later classifier, libraries from the same donor must not be split between training and test sets. The X/Y-excluded matrix is only a candidate feature universe; feature selection, scaling and other model-tuned preprocessing must be performed within the training data / cross-validation folds.

## 11. Save reusable outputs and a Part 1 summary
The following outputs are saved so later notebook sections can load the processed data without repeating the normalization step:
- cleaned sample metadata;
- the main low-expression-filtered raw integer counts (all supplied genes; used for the Part 1 normalization/VST/PCA);
- a separate low-expression-filtered raw integer count matrix excluding chromosome X/Y genes (for later genome-wide analyses that follow the paper's feature scope);
- VST-normalized expression for Part 1 exploration (all retained genes);
- an X/Y-excluded row-subset of the VST matrix for later exploratory feature use (not a fold-wise classifier preprocessing pipeline);
- size factors and sample QC metrics, including the exploratory mitochondrial gene-count fraction;
- the count-matrix-matched Ensembl GRCh37 release 75 gene annotation;
- an annotation table listing the filtered X/Y genes excluded from the separate downstream matrix;
- PCA coordinates and donor-level summary tables.

*The original count matrix and downloaded GTF are kept unchanged in the folder `data`. The separate X/Y-excluded matrix does not replace the main Part 1 matrix.*

In [ ]:
# Saves processed matrices and metadata using pickle to preserve pandas indexes and data types.
meta.to_pickle(PROCESSED_DIR / "01_metadata_clean.pkl")
counts_filtered.to_pickle(PROCESSED_DIR / "01_filtered_raw_counts.pkl")
vst.to_pickle(PROCESSED_DIR / "01_vst_expression.pkl")

vst_no_sex_chr.to_pickle(
    PROCESSED_DIR / "01_vst_expression_no_XY.pkl"
)

counts_filtered_no_sex_chr.to_pickle(
    PROCESSED_DIR / "01_filtered_raw_counts_no_XY.pkl"
)
excluded_sex_chromosome_annotation.to_csv(
    TABLE_DIR / "01_excluded_XY_gene_annotation.csv",
    index_label="gene_id",
)

# Saves gene annotation and the exploratory mitochondrial gene-count fraction.
gene_annotation.to_csv(PROCESSED_DIR / "01_gene_annotation_matched.csv", index_label="gene_id")
mt_gene_count_fraction.to_frame().to_csv(
    PROCESSED_DIR / "01_mt_gene_count_fraction.csv", index_label="library_id"
)

# Stores a compact summary of the Part 1 run.
part1_summary = pd.Series({
    "n_genes_raw": int(counts.shape[0]),
    "n_genes_filtered": int(counts_filtered.shape[0]),
    "n_X_genes_in_input_matrix": int(gene_annotation["chr"].eq("X").sum()),
    "n_Y_genes_in_input_matrix": int(gene_annotation["chr"].eq("Y").sum()),
    "n_filtered_XY_genes_excluded_for_downstream": int(len(sex_chromosome_gene_ids_filtered)),
    "n_genes_filtered_no_XY": int(counts_filtered_no_sex_chr.shape[0]),
    "n_vst_genes_no_XY": int(vst_no_sex_chr.shape[0]),
    "n_libraries": int(counts.shape[1]),
    "n_annotated_genes": int(gene_annotation.shape[0]),
    "n_mitochondrial_genes": int(len(mt_gene_ids)),
    "median_mt_gene_count_fraction": float(mt_gene_count_fraction.median()),
    "n_donors": int(meta["donor_id"].nunique()),
    "n_libraries_missing_pH": int(meta["brain_pH"].isna().sum()),
    "n_donors_missing_pH": int(meta.loc[meta["brain_pH"].isna(), "donor_id"].nunique()),
    "n_qc_flagged_libraries": int(sample_qc["qc_flag_for_inspection"].sum()),
    "min_total_gene_counts": int(sample_qc["total_gene_counts"].min()),
    "median_total_gene_counts": float(sample_qc["total_gene_counts"].median()),
    "max_total_gene_counts": int(sample_qc["total_gene_counts"].max()),
    "median_detected_genes": float(sample_qc["detected_genes"].median()),
    "egr1_ensembl_id_present": bool(EGR1_ENSEMBL_ID in counts.index),
    "egr1_passes_filter": bool(EGR1_ENSEMBL_ID in counts_filtered.index),
    "pc1_variance_percent": float(explained_variance_percent[0]),
    "pc2_variance_percent": float(explained_variance_percent[1]),
    "pydeseq2_version": PYDESEQ2_VERSION,
    "python_version": sys.version.split()[0],
})

part1_summary.to_csv(TABLE_DIR / "01_part1_summary.csv", header=["value"])

print("PART 1 SUMMARY")
print(part1_summary.to_string())

print("\nSaved processed files:")
for path in sorted(PROCESSED_DIR.glob("*")):
    print(f"  {path} ({path.stat().st_size / (1024 ** 2):.1f} MB)")

print("\nSaved figures:")
for path in sorted(FIGURE_DIR.glob("*")):
    print(f"  {path}")

print("\nSaved tables:")
for path in sorted(TABLE_DIR.glob("*")):
    print(f"  {path}")

---

___

# <mark style="background-color: lightblue">Part 2 - Formulating and testing a research question</mark>

Part 2 tests whether EGR1 expression differs between the four diagnostic groups after accounting for relevant sample and donor characteristics.

This section reuses the normalized, variance-stabilized EGR1 expression calculated in Part 1. It does not repeat the genome-wide preprocessing. The analysis is limited to one pre-specified gene (EGR1), so no genome-wide differential-expression procedure is performed here.

**Important design feature:** the 281 libraries come from 96 donors, and most donors contributed more than one brain region. Libraries from the same donor are therefore not independent. The statistical analysis uses a donor-level random intercept to account for this repeated-measures structure.

## 2.1 The research question and hypotheses

### Research question

**How does normalized EGR1 expression vary across Control, Schizophrenia, Bipolar Disorder and Major Depression after accounting for brain region, brain pH, age, sex and post-mortem interval, while accounting for repeated measurements from the same donor?**